# 参考题解：带 KV Cache 的多头注意力

实现支持增量解码的多头自注意力。`use_cache=True` 时连续调用应不断增长 K/V 缓存。

核心思路：启用缓存时把本轮 K/V 追加到历史缓存，Q 只保留本轮查询；每次追加后必须更新 cache_k 和 cache_v。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import math

class CachedMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.reset_cache()

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None

    def forward(self, x: torch.Tensor, mask=None, use_cache: bool = False):
        batch, query_len, _ = x.shape
        q = self.W_q(x).view(batch, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(batch, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(batch, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        if use_cache:
            if self.cache_k is not None:
                k = torch.cat([self.cache_k, k], dim=2)
                v = torch.cat([self.cache_v, v], dim=2)
            self.cache_k, self.cache_v = k, v
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).contiguous().view(batch, query_len, self.d_model)
        return self.W_o(out)
